# 16 — Validate canonical CNV consensus

This notebook reproduces the historical final visual QC and statistical validation of the marker + CopyKAT + infercnvpy CNV consensus.

## Historical semantics preserved

The statistical procedures are unchanged from the historical executed notebook:

- cell-level CopyKAT composition summaries;
- UMAP visual QC;
- chi-square association tests with Cramer's V;
- per-cluster and per-final-group Fisher exact enrichment tests;
- tumor/cycling versus pooled reference-cluster Fisher tests;
- sample-level condition comparisons using Kruskal-Wallis and pairwise Mann-Whitney U tests;
- Benjamini-Hochberg FDR correction.

The historical decision sets are unchanged:

- malignant clusters: `0, 4, 7, 11`;
- cycling candidate cluster: `2`;
- core reference clusters: `8, 14`.

## Canonical input policy

This notebook requires the standard final H5AD produced by Notebook 15:

`GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad`

It does not fall back to a pre-consensus H5AD. Before statistical analysis, the notebook verifies the canonical historical CopyKAT counts and the historical final consensus cell counts.

## Memory-safety policy

The H5AD is opened in backed mode. Only `.obs` and `X_umap` are materialized, after which the file handle is closed. The full expression matrix is never loaded into RAM.

Hard checkpoints are included for the historical output-table shapes and key cell counts. Numerical tests retain the historical code unchanged.


## Pandas categorical compatibility

`clean_copykat_pred()` is a vectorized Series-normalization helper. Canonical CopyKAT predictions may be stored as a pandas categorical column in the H5AD, so this notebook applies the helper to the full Series rather than through `Series.map()`. This is an implementation compatibility fix only; no statistical procedure or biological rule is changed.


In [1]:
# =========================
# CONFIG
# =========================
from pathlib import Path
import os
import re
import math
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

from scipy.stats import fisher_exact, chi2_contingency, kruskal, mannwhitneyu

try:
    import anndata as ad
except Exception as e:
    raise ImportError("Please install/load anndata in this Python environment.") from e

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_DIR / "metadata"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed" / "integrated"
FIG_DIR = PROJECT_DIR / "figures" / "final_CNV_QC_HVG8000_r0_6"
FIG_DIR.mkdir(parents=True, exist_ok=True)
METADATA_DIR.mkdir(parents=True, exist_ok=True)

RUN_TAG = "HVG8000_r0_6"

FINAL_H5AD = (
    PROCESSED_DIR
    / "GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad"
)

if not FINAL_H5AD.exists():
    raise FileNotFoundError(
        "Notebook 16 requires the canonical Notebook-15 final consensus H5AD:\n"
        f"{FINAL_H5AD}\n"
        "Run Notebook 15 successfully before Notebook 16."
    )

CLUSTER_CONSENSUS_TSV = METADATA_DIR / f"final_copykat_marker_infercnvpy_cluster_consensus_{RUN_TAG}.tsv"
CLUSTER_COUNTS_TSV = METADATA_DIR / f"final_copykat_marker_cluster_counts_{RUN_TAG}.tsv"

# To keep plotting fast and memory-safe.
# Set to None if you want all cells plotted.
PLOT_MAX_POINTS = 250_000
RANDOM_SEED = 17

# Main clusters based on our final decision.
MALIGNANT_CLUSTERS = {"0", "4", "7", "11"}
CYCLING_CANDIDATE_CLUSTERS = {"2"}
CORE_REFERENCE_CLUSTERS = {"8", "14"}

print("Project:", PROJECT_DIR)
print("Figures:", FIG_DIR)


Project: G:\Repository\glioma-cnv-single-cell-atlas
Figures: G:\Repository\glioma-cnv-single-cell-atlas\figures\final_CNV_QC_HVG8000_r0_6


In [2]:
# =========================
# HELPERS
# =========================
def first_existing(paths, label="file"):
    existing = [Path(p) for p in paths if Path(p).exists()]
    if not existing:
        raise FileNotFoundError("No %s found. Checked:\n" % label + "\n".join(str(p) for p in paths))
    print(f"[USING {label}]", existing[0])
    return existing[0]


def choose_col(df, candidates, required=True, label="column"):
    for c in candidates:
        if c in df.columns:
            print(f"[USING {label}] {c}")
            return c
    if required:
        raise KeyError(f"None of these {label} candidates found: {candidates}\nAvailable columns: {list(df.columns)[:120]}")
    print(f"[MISSING optional {label}] candidates: {candidates}")
    return None


def write_tsv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")
    return path


def bh_fdr(pvals):
    p = np.asarray([np.nan if x is None else x for x in pvals], dtype=float)
    out = np.full_like(p, np.nan, dtype=float)
    valid = np.isfinite(p)
    if valid.sum() == 0:
        return out
    pv = p[valid]
    order = np.argsort(pv)
    ranked = pv[order]
    n = len(ranked)
    q = ranked * n / (np.arange(1, n + 1))
    q = np.minimum.accumulate(q[::-1])[::-1]
    q = np.clip(q, 0, 1)
    restored = np.empty_like(q)
    restored[order] = q
    out[valid] = restored
    return out


def as_cluster_str(s):
    x = pd.Series(s).astype(str)
    x = x.str.replace(r"\.0$", "", regex=True)
    return x


def clean_copykat_pred(s):
    x = pd.Series(s).astype(str).str.strip().str.lower()
    x = x.replace({
        "not.defined": "not_defined",
        "not defined": "not_defined",
        "not_defined": "not_defined",
        "nan": "missing",
        "none": "missing",
        "": "missing",
    })
    x = x.where(x.isin(["aneuploid", "diploid", "not_defined", "missing"]), x)
    return x


def safe_odds_ratio(a, b, c, d):
    # Haldane-Anscombe only for display if zero cells exist.
    if min(a, b, c, d) == 0:
        return ((a + 0.5) * (d + 0.5)) / ((b + 0.5) * (c + 0.5))
    return (a * d) / (b * c)


def cramer_v_from_table(tbl):
    chi2, p, dof, expected = chi2_contingency(tbl)
    n = np.asarray(tbl).sum()
    r, k = np.asarray(tbl).shape
    if n == 0 or min(r - 1, k - 1) <= 0:
        return np.nan, chi2, p, dof
    phi2 = chi2 / n
    return math.sqrt(phi2 / min(k - 1, r - 1)), chi2, p, dof


## 1. Load final consensus h5ad and merge cluster-level consensus if needed

In [3]:
# Open only in backed mode and materialize metadata + UMAP.
adata = ad.read_h5ad(
    FINAL_H5AD,
    backed="r",
)

try:
    source_shape = adata.shape
    obs = adata.obs.copy()
    obs.index = obs.index.astype(str)

    obsm_keys = list(
        adata.obsm.keys()
    )

    if "X_umap" in obsm_keys:
        umap = pd.DataFrame(
            np.asarray(
                adata.obsm["X_umap"]
            ).copy(),
            index=obs.index,
            columns=[
                "UMAP1",
                "UMAP2",
            ],
        )
    else:
        umap = None

finally:
    adata.file.close()

print(
    "Final canonical H5AD:",
    FINAL_H5AD,
)
print(
    "source shape:",
    source_shape,
)
print(
    "obs shape:",
    obs.shape,
)
print(
    "obsm keys:",
    obsm_keys,
)

if source_shape != (
    351427,
    15176,
):
    raise RuntimeError(
        f"Canonical Notebook-15 H5AD shape mismatch: {source_shape}"
    )

# Detect main columns.
cluster_col = choose_col(obs, [
    "leiden_scvi_main",
    "leiden_scvi_HVG8000_r0_6",
    "leiden_scvi_0.6",
    "leiden_scvi_0.7",
    "cluster",
], label="cluster")

annotation_col = choose_col(obs, [
    "annotation_marker_based_HVG8000_r0_6",
    "annotation_marker_based",
    "cell_type_marker_based",
    "annotation",
], required=False, label="annotation")

copykat_col = choose_col(obs, [
    "copykat_pred_clean",
    "copykat_microbatch_pred_clean",
    "copykat_prediction_clean",
    "copykat.pred",
    "copykat_pred",
    "copykat_call",
], required=False, label="CopyKAT prediction")

final_group_col = choose_col(obs, [
    "final_malignancy_group_copykat_marker",
    "final_malignancy_group",
    "malignancy_group_copykat_marker",
    "marker_copykat_malignancy_group",
], required=False, label="final malignancy group")

condition_col = choose_col(obs, ["condition", "disease", "diagnosis", "tumor_status"], required=False, label="condition")
dataset_col = choose_col(obs, ["core_dataset", "dataset_id", "dataset", "Dataset"], required=False, label="dataset")
patient_col = choose_col(obs, ["patient_id", "patient", "Patient"], required=False, label="patient")
sample_col = choose_col(obs, ["sample_id", "sample", "Sample", "orig.ident"], required=False, label="sample")

# Standardized analysis columns.
obs["cluster_str"] = as_cluster_str(obs[cluster_col])

# Merge final cluster consensus if final group or CopyKAT column is missing/incomplete.
if CLUSTER_COUNTS_TSV.exists():
    cluster_counts = pd.read_csv(CLUSTER_COUNTS_TSV, sep="\t")
    cluster_counts["cluster_str"] = as_cluster_str(cluster_counts["cluster"])
    merge_cols = ["cluster_str"]
    for c in ["final_malignancy_group_copykat_marker", "final_cnv_consensus_by_cluster", "aneuploid_fraction_defined", "aneuploid_fraction_all"]:
        if c in cluster_counts.columns and c not in obs.columns:
            merge_cols.append(c)
    if len(merge_cols) > 1:
        obs = obs.merge(cluster_counts[merge_cols].drop_duplicates("cluster_str"), on="cluster_str", how="left", suffixes=("", "_from_cluster"))
        print("[MERGED cluster_counts]", merge_cols)

# Re-detect after merge.
if final_group_col is None:
    final_group_col = choose_col(obs, ["final_malignancy_group_copykat_marker"], label="final malignancy group after merge")
if copykat_col is None:
    copykat_col = choose_col(obs, ["copykat_pred_clean", "copykat_microbatch_pred_clean", "copykat_prediction_clean"], required=False, label="CopyKAT prediction after merge")

# If CopyKAT pred column is still missing but counts only exist, stop because cell-level stats need predictions.
if copykat_col is None:
    raise KeyError("No cell-level CopyKAT prediction column found in final h5ad. Please check final consensus h5ad obs columns.")

obs["copykat_pred_clean_for_stats"] = clean_copykat_pred(obs[copykat_col])
obs["final_group_for_stats"] = obs[final_group_col].astype(str).fillna("missing")


# UMAP was copied while the backed file was open.
if umap is not None:
    print(
        "UMAP shape:",
        umap.shape,
    )

    if umap.shape != (
        351427,
        2,
    ):
        raise RuntimeError(
            f"Canonical UMAP shape mismatch: {umap.shape}"
        )

else:
    print(
        "[WARNING] X_umap not found; UMAP plots will be skipped."
    )

# Canonical Notebook-15 hard checkpoints.
# clean_copykat_pred is intentionally vectorized and accepts a full Series.
# Do not pass it through Series.map(), which would feed scalar category
# values into the vectorized function and can fail for pandas Categoricals.
observed_copykat = (
    clean_copykat_pred(
        obs[
            copykat_col
        ]
    )
    .value_counts(
        dropna=False
    )
    .to_dict()
)

EXPECTED_COPYKAT_COUNTS = {
    "aneuploid": 202895,
    "diploid": 111493,
    "not_defined": 37037,
    "missing": 2,
}

observed_copykat = {
    str(key): int(value)
    for key, value
    in observed_copykat.items()
}

if observed_copykat != EXPECTED_COPYKAT_COUNTS:
    raise RuntimeError(
        "Notebook 16 is not consuming the exact canonical historical "
        f"CopyKAT calls. Observed: {observed_copykat}"
    )

EXPECTED_FINAL_GROUP_COUNTS = {
    "review": 167178,
    "malignant_like": 123250,
    "cycling_malignant_like_candidate": 38634,
    "non_malignant_or_diploid_leaning": 22365,
}

observed_final_groups = {
    str(key): int(value)
    for key, value
    in obs[
        final_group_col
    ]
    .astype(str)
    .value_counts(
        dropna=False
    )
    .items()
}

if observed_final_groups != EXPECTED_FINAL_GROUP_COUNTS:
    raise RuntimeError(
        "Notebook-15 final malignancy-group checkpoint failed. "
        f"Observed: {observed_final_groups}"
    )

print(
    "PASS: canonical Notebook-15 CopyKAT and final-group checkpoints match exactly."
)

column_report = pd.DataFrame([
    {"field": "cluster", "column": cluster_col},
    {"field": "annotation", "column": annotation_col},
    {"field": "copykat_prediction", "column": copykat_col},
    {"field": "final_malignancy_group", "column": final_group_col},
    {"field": "condition", "column": condition_col},
    {"field": "dataset", "column": dataset_col},
    {"field": "patient", "column": patient_col},
    {"field": "sample", "column": sample_col},
])
write_tsv(column_report, METADATA_DIR / f"final_CNV_QC_obs_column_report_{RUN_TAG}.tsv")

readiness = pd.DataFrame([{
    "final_h5ad": str(FINAL_H5AD),
    "n_cells": obs.shape[0],
    "n_obs_columns": obs.shape[1],
    "has_umap": umap is not None,
    "cluster_col": cluster_col,
    "copykat_col": copykat_col,
    "final_group_col": final_group_col,
}])
write_tsv(readiness, METADATA_DIR / f"final_CNV_QC_readiness_{RUN_TAG}.tsv")


Final canonical H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad
source shape: (351427, 15176)
obs shape: (351427, 88)
obsm keys: ['X_scVI', 'X_umap']
[USING cluster] leiden_scvi_main
[USING annotation] annotation_marker_based_HVG8000_r0_6
[USING CopyKAT prediction] copykat_pred_clean
[USING final malignancy group] final_malignancy_group_copykat_marker
[USING condition] condition
[USING dataset] core_dataset
[USING patient] patient_id
[USING sample] sample_id
[MERGED cluster_counts] ['cluster_str', 'aneuploid_fraction_defined', 'aneuploid_fraction_all']
UMAP shape: (351427, 2)
PASS: canonical Notebook-15 CopyKAT and final-group checkpoints match exactly.
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_CNV_QC_obs_column_report_HVG8000_r0_6.tsv shape=(8, 2)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_CNV_QC_readiness_HVG8000_r0_6.tsv sha

WindowsPath('G:/Repository/glioma-cnv-single-cell-atlas/metadata/final_CNV_QC_readiness_HVG8000_r0_6.tsv')

## 2. Final counts and composition summaries

In [4]:
# Overall final group and CopyKAT composition.
group_counts = (
    obs.groupby(["final_group_for_stats", "copykat_pred_clean_for_stats"], observed=True)
       .size().reset_index(name="n_cells")
       .sort_values(["final_group_for_stats", "n_cells"], ascending=[True, False])
)
group_counts["fraction_all_cells"] = group_counts["n_cells"] / len(obs)
write_tsv(group_counts, METADATA_DIR / f"final_CNV_QC_group_counts_{RUN_TAG}.tsv")

def summarize_cluster_group(df):
    g = (df.groupby(["cluster_str", "final_group_for_stats", "copykat_pred_clean_for_stats"], observed=True)
           .size().reset_index(name="n_cells"))
    # Wide for useful metrics.
    wide = g.pivot_table(index=["cluster_str", "final_group_for_stats"], columns="copykat_pred_clean_for_stats", values="n_cells", fill_value=0, aggfunc="sum").reset_index()
    for c in ["aneuploid", "diploid", "not_defined", "missing"]:
        if c not in wide.columns:
            wide[c] = 0
    wide["n_defined"] = wide["aneuploid"] + wide["diploid"]
    wide["n_total"] = wide[["aneuploid", "diploid", "not_defined", "missing"]].sum(axis=1)
    wide["aneuploid_fraction_defined"] = np.where(wide["n_defined"] > 0, wide["aneuploid"] / wide["n_defined"], np.nan)
    wide["aneuploid_fraction_all"] = np.where(wide["n_total"] > 0, wide["aneuploid"] / wide["n_total"], np.nan)
    return wide.sort_values("cluster_str", key=lambda s: pd.to_numeric(s, errors="coerce"))

cluster_group_summary = summarize_cluster_group(obs)
write_tsv(cluster_group_summary, METADATA_DIR / f"final_CNV_QC_cluster_group_copykat_summary_{RUN_TAG}.tsv")

display(group_counts)
display(cluster_group_summary.head(25))

EXPECTED_COMPOSITION_SHAPES = {
    "group_counts": (14, 4),
    "cluster_group_summary": (18, 10),
}

observed_composition_shapes = {
    "group_counts": group_counts.shape,
    "cluster_group_summary": cluster_group_summary.shape,
}

if observed_composition_shapes != EXPECTED_COMPOSITION_SHAPES:
    raise RuntimeError(
        "Historical Notebook-16 composition-shape checkpoint failed. "
        f"Observed: {observed_composition_shapes}"
    )

print(
    "PASS: historical composition-table shapes match exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_CNV_QC_group_counts_HVG8000_r0_6.tsv shape=(14, 4)
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_CNV_QC_cluster_group_copykat_summary_HVG8000_r0_6.tsv shape=(18, 10)


,final_group_for_stats,copykat_pred_clean_for_stats,n_cells,fraction_all_cells
0,cycling_malignant_like_candidate,aneuploid,29096,0.082794
1,cycling_malignant_like_candidate,diploid,5913,0.016826
2,cycling_malignant_like_candidate,not_defined,3625,0.010315
3,malignant_like,aneuploid,88602,0.252121
4,malignant_like,diploid,20033,0.057005
6,malignant_like,not_defined,14614,0.041585
5,malignant_like,missing,1,0.000003
8,non_malignant_or_diploid_leaning,diploid,16139,0.045924
7,non_malignant_or_diploid_leaning,aneuploid,4379,0.012461
9,non_malignant_or_diploid_leaning,not_defined,1847,0.005256


copykat_pred_clean_for_stats,cluster_str,final_group_for_stats,aneuploid,diploid,missing,not_defined,n_defined,n_total,aneuploid_fraction_defined,aneuploid_fraction_all
0,0,malignant_like,45910,8947,0,6789,54857,61646,0.836903,0.744736
1,1,review,29402,24843,0,5546,54245,59791,0.542022,0.491746
10,2,cycling_malignant_like_candidate,29096,5913,0,3625,35009,38634,0.831101,0.753119
11,3,review,21350,12033,1,1971,33383,35355,0.639547,0.603875
12,4,malignant_like,24754,4562,0,3580,29316,32896,0.844385,0.752493
13,5,review,8354,12387,0,3236,20741,23977,0.402777,0.348417
14,6,review,10847,10728,0,1368,21575,22943,0.502758,0.472780
15,7,malignant_like,14292,5140,1,3092,19432,22525,0.735488,0.634495
16,8,non_malignant_or_diploid_leaning,2787,11317,0,705,14104,14809,0.197604,0.188196
17,9,review,3400,6255,0,4209,9655,13864,0.352149,0.245239


PASS: historical composition-table shapes match exactly.


## 3. Final UMAP visual QC

In [5]:
figure_records = []


def plot_umap_categorical(obs_df, umap_df, color_col, title, out_png, max_points=PLOT_MAX_POINTS, seed=RANDOM_SEED):
    if umap_df is None:
        return None
    df = umap_df.join(obs_df[[color_col]].astype(str), how="inner")
    df = df.replace({color_col: {"nan": "missing", "None": "missing"}})
    if max_points is not None and len(df) > max_points:
        df = df.sample(max_points, random_state=seed)
    cats = pd.Categorical(df[color_col]).categories.tolist()
    cmap = plt.get_cmap("tab20") if len(cats) <= 20 else plt.get_cmap("hsv")
    color_map = {cat: cmap(i / max(1, len(cats)-1)) for i, cat in enumerate(cats)}
    fig, ax = plt.subplots(figsize=(9, 7), dpi=160)
    for cat in cats:
        sub = df[df[color_col] == cat]
        ax.scatter(sub["UMAP1"], sub["UMAP2"], s=0.25, alpha=0.65, label=str(cat), color=color_map[cat], linewidths=0)
    ax.set_title(title)
    ax.set_xlabel("UMAP1")
    ax.set_ylabel("UMAP2")
    ax.set_xticks([])
    ax.set_yticks([])
    # Keep legend readable.
    if len(cats) <= 30:
        handles = [Line2D([0], [0], marker='o', color='w', markerfacecolor=color_map[c], markersize=5, label=str(c)) for c in cats]
        ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.02, 0.5), frameon=False, fontsize=7)
    fig.tight_layout()
    fig.savefig(out_png, bbox_inches="tight")
    plt.close(fig)
    figure_records.append({"figure_label": title, "path": str(out_png), "exists": Path(out_png).exists(), "size_mb": Path(out_png).stat().st_size/1024/1024 if Path(out_png).exists() else np.nan})
    print("[SAVED FIG]", out_png)
    return out_png

# Temporary plotting columns.
plot_obs = obs.copy()
plot_obs["cluster_for_plot"] = plot_obs["cluster_str"]
plot_obs["copykat_for_plot"] = plot_obs["copykat_pred_clean_for_stats"]
plot_obs["final_group_for_plot"] = plot_obs["final_group_for_stats"]

plot_items = [
    ("final_group_for_plot", "Final malignancy group + CopyKAT/marker consensus"),
    ("copykat_for_plot", "CopyKAT prediction"),
    ("cluster_for_plot", "scVI Leiden cluster"),
]
if annotation_col is not None:
    plot_items.append((annotation_col, "Marker-based annotation"))
if condition_col is not None:
    plot_items.append((condition_col, "Condition"))
if dataset_col is not None:
    plot_items.append((dataset_col, "Dataset"))

for col, title in plot_items:
    safe_name = re.sub(r"[^A-Za-z0-9_]+", "_", col)
    plot_umap_categorical(
        plot_obs,
        umap,
        col,
        title,
        FIG_DIR / f"UMAP_{safe_name}_{RUN_TAG}.png",
    )

# Barplot final group counts.
fg = obs["final_group_for_stats"].value_counts().reset_index()
fg.columns = ["final_group", "n_cells"]
fig, ax = plt.subplots(figsize=(8, 4), dpi=160)
ax.bar(fg["final_group"].astype(str), fg["n_cells"])
ax.set_ylabel("Cells")
ax.set_title("Final malignancy group counts")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
out = FIG_DIR / f"bar_final_malignancy_group_counts_{RUN_TAG}.png"
fig.savefig(out, bbox_inches="tight")
plt.close(fig)
figure_records.append({"figure_label": "bar_final_malignancy_group_counts", "path": str(out), "exists": out.exists(), "size_mb": out.stat().st_size/1024/1024 if out.exists() else np.nan})

figure_manifest = pd.DataFrame(figure_records)
write_tsv(figure_manifest, METADATA_DIR / f"final_CNV_QC_figure_manifest_{RUN_TAG}.tsv")
display(figure_manifest)

if umap is not None:
    if figure_manifest.shape != (7, 4):
        raise RuntimeError(
            "Historical Notebook-16 figure-manifest checkpoint failed: "
            f"{figure_manifest.shape} != (7, 4)"
        )

    print(
        "PASS: historical figure-manifest shape matches exactly."
    )


[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_CNV_QC_HVG8000_r0_6\UMAP_final_group_for_plot_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_CNV_QC_HVG8000_r0_6\UMAP_copykat_for_plot_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_CNV_QC_HVG8000_r0_6\UMAP_cluster_for_plot_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_CNV_QC_HVG8000_r0_6\UMAP_annotation_marker_based_HVG8000_r0_6_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_CNV_QC_HVG8000_r0_6\UMAP_condition_HVG8000_r0_6.png
[SAVED FIG] G:\Repository\glioma-cnv-single-cell-atlas\figures\final_CNV_QC_HVG8000_r0_6\UMAP_core_dataset_HVG8000_r0_6.png
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_CNV_QC_figure_manifest_HVG8000_r0_6.tsv shape=(7, 4)


,figure_label,path,exists,size_mb
0,Final malignancy group + CopyKAT/marker consensus,G:\Repository\glioma-cnv-single-cell-atlas\fig...,True,0.020896
1,CopyKAT prediction,G:\Repository\glioma-cnv-single-cell-atlas\fig...,True,0.016543
2,scVI Leiden cluster,G:\Repository\glioma-cnv-single-cell-atlas\fig...,True,0.015705
3,Marker-based annotation,G:\Repository\glioma-cnv-single-cell-atlas\fig...,True,0.016866
4,Condition,G:\Repository\glioma-cnv-single-cell-atlas\fig...,True,0.014236
5,Dataset,G:\Repository\glioma-cnv-single-cell-atlas\fig...,True,0.014087
6,bar_final_malignancy_group_counts,G:\Repository\glioma-cnv-single-cell-atlas\fig...,True,0.048869


PASS: historical figure-manifest shape matches exactly.


## 4. Statistical validation: CopyKAT enrichment by cluster and final group

These tests use only defined CopyKAT cells (`aneuploid` and `diploid`) and ignore `not_defined/missing` for Fisher tests.

In [6]:
defined = obs[obs["copykat_pred_clean_for_stats"].isin(["aneuploid", "diploid"])].copy()
print("Defined CopyKAT cells:", defined.shape[0], "of", obs.shape[0])

# Overall association tests.
overall_records = []
for label, col in [
    ("cluster_vs_copykat", "cluster_str"),
    ("final_group_vs_copykat", "final_group_for_stats"),
]:
    tbl = pd.crosstab(defined[col], defined["copykat_pred_clean_for_stats"])
    # Ensure both columns present.
    for c in ["aneuploid", "diploid"]:
        if c not in tbl.columns:
            tbl[c] = 0
    tbl = tbl[["aneuploid", "diploid"]]
    if tbl.shape[0] >= 2 and tbl.values.sum() > 0:
        cv, chi2, p, dof = cramer_v_from_table(tbl.values)
    else:
        cv, chi2, p, dof = np.nan, np.nan, np.nan, np.nan
    overall_records.append({
        "test": label,
        "n_cells_defined": int(tbl.values.sum()),
        "n_categories": tbl.shape[0],
        "chi2": chi2,
        "dof": dof,
        "p_value": p,
        "cramers_v": cv,
    })

overall_tests = pd.DataFrame(overall_records)
overall_tests["q_value_bh"] = bh_fdr(overall_tests["p_value"])
write_tsv(overall_tests, METADATA_DIR / f"final_stat_overall_association_tests_{RUN_TAG}.tsv")
display(overall_tests)

# Per-cluster enrichment: cluster vs all other defined cells.
records = []
all_a = int((defined["copykat_pred_clean_for_stats"] == "aneuploid").sum())
all_d = int((defined["copykat_pred_clean_for_stats"] == "diploid").sum())
for cl, sub in defined.groupby("cluster_str", observed=True):
    a = int((sub["copykat_pred_clean_for_stats"] == "aneuploid").sum())
    d = int((sub["copykat_pred_clean_for_stats"] == "diploid").sum())
    c = all_a - a
    b = all_d - d
    # table: target aneup/dip vs rest aneup/dip
    odds, p = fisher_exact([[a, d], [c, b]], alternative="two-sided")
    records.append({
        "cluster": cl,
        "n_defined": a + d,
        "n_aneuploid": a,
        "n_diploid": d,
        "aneuploid_fraction_defined": a/(a+d) if (a+d)>0 else np.nan,
        "rest_n_aneuploid": c,
        "rest_n_diploid": b,
        "odds_ratio_cluster_vs_rest": safe_odds_ratio(a, d, c, b),
        "fisher_p_value": p,
    })
cluster_enrichment = pd.DataFrame(records)
cluster_enrichment["fisher_q_value_bh"] = bh_fdr(cluster_enrichment["fisher_p_value"])
cluster_enrichment = cluster_enrichment.sort_values("cluster", key=lambda s: pd.to_numeric(s, errors="coerce"))
write_tsv(cluster_enrichment, METADATA_DIR / f"final_stat_cluster_copykat_enrichment_{RUN_TAG}.tsv")
display(cluster_enrichment)

# Final group enrichment.
records = []
for gr, sub in defined.groupby("final_group_for_stats", observed=True):
    a = int((sub["copykat_pred_clean_for_stats"] == "aneuploid").sum())
    d = int((sub["copykat_pred_clean_for_stats"] == "diploid").sum())
    c = all_a - a
    b = all_d - d
    odds, p = fisher_exact([[a, d], [c, b]], alternative="two-sided")
    records.append({
        "final_group": gr,
        "n_defined": a + d,
        "n_aneuploid": a,
        "n_diploid": d,
        "aneuploid_fraction_defined": a/(a+d) if (a+d)>0 else np.nan,
        "odds_ratio_group_vs_rest": safe_odds_ratio(a, d, c, b),
        "fisher_p_value": p,
    })
group_enrichment = pd.DataFrame(records)
group_enrichment["fisher_q_value_bh"] = bh_fdr(group_enrichment["fisher_p_value"])
group_enrichment = group_enrichment.sort_values("aneuploid_fraction_defined", ascending=False)
write_tsv(group_enrichment, METADATA_DIR / f"final_stat_group_copykat_enrichment_{RUN_TAG}.tsv")
display(group_enrichment)

EXPECTED_DEFINED_COPYKAT_CELLS = 314388

if len(defined) != EXPECTED_DEFINED_COPYKAT_CELLS:
    raise RuntimeError(
        "Historical defined-CopyKAT cell-count checkpoint failed: "
        f"{len(defined)} != {EXPECTED_DEFINED_COPYKAT_CELLS}"
    )

EXPECTED_ASSOCIATION_SHAPES = {
    "overall_tests": (2, 8),
    "cluster_enrichment": (18, 10),
    "group_enrichment": (4, 8),
}

observed_association_shapes = {
    "overall_tests": overall_tests.shape,
    "cluster_enrichment": cluster_enrichment.shape,
    "group_enrichment": group_enrichment.shape,
}

if observed_association_shapes != EXPECTED_ASSOCIATION_SHAPES:
    raise RuntimeError(
        "Historical Notebook-16 association-table shape checkpoint failed. "
        f"Observed: {observed_association_shapes}"
    )

print(
    "PASS: historical defined-cell count and association-table shapes match exactly."
)


Defined CopyKAT cells: 314388 of 351427
[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_stat_overall_association_tests_HVG8000_r0_6.tsv shape=(2, 8)


,test,n_cells_defined,n_categories,chi2,dof,p_value,cramers_v,q_value_bh
0,cluster_vs_copykat,314388,18,50389.907465,17,0.0,0.400349,0.0
1,final_group_vs_copykat,314388,4,43327.504327,3,0.0,0.371235,0.0


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_stat_cluster_copykat_enrichment_HVG8000_r0_6.tsv shape=(18, 10)


,cluster,n_defined,n_aneuploid,n_diploid,aneuploid_fraction_defined,rest_n_aneuploid,rest_n_diploid,odds_ratio_cluster_vs_rest,fisher_p_value,fisher_q_value_bh
0,0,54857,45910,8947,0.836903,156985,102546,3.351895,0.000000e+00,0.000000e+00
1,1,54245,29402,24843,0.542022,173493,86650,0.591098,0.000000e+00,0.000000e+00
10,2,35009,29096,5913,0.831101,173799,105580,2.989233,0.000000e+00,0.000000e+00
11,3,33383,21350,12033,0.639547,181545,99460,0.972049,1.889548e-02,1.889548e-02
12,4,29316,24754,4562,0.844385,178141,106931,3.257091,0.000000e+00,0.000000e+00
13,5,20741,8354,12387,0.402777,194541,99106,0.343572,0.000000e+00,0.000000e+00
14,6,21575,10847,10728,0.502758,192048,100765,0.530507,0.000000e+00,0.000000e+00
15,7,19432,14292,5140,0.735488,188603,106353,1.567946,7.408235e-169,1.212257e-168
16,8,14104,2787,11317,0.197604,200108,100176,0.123283,0.000000e+00,0.000000e+00
17,9,9655,3400,6255,0.352149,199495,105238,0.286743,0.000000e+00,0.000000e+00


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_stat_group_copykat_enrichment_HVG8000_r0_6.tsv shape=(4, 8)


,final_group,n_defined,n_aneuploid,n_diploid,aneuploid_fraction_defined,odds_ratio_group_vs_rest,fisher_p_value,fisher_q_value_bh
0,cycling_malignant_like_candidate,35009,29096,5913,0.831101,2.989233,0.0,0.0
1,malignant_like,108635,88602,20033,0.815594,3.539233,0.0,0.0
3,review,150226,80818,69408,0.537976,0.401414,0.0,0.0
2,non_malignant_or_diploid_leaning,20518,4379,16139,0.213422,0.130329,0.0,0.0


PASS: historical defined-cell count and association-table shapes match exactly.


## 5. Statistical validation: tumor-like and cycling clusters versus core reference clusters

In [7]:
# Tumor/cycling clusters versus pooled core reference clusters.
ref_mask = defined["cluster_str"].isin(CORE_REFERENCE_CLUSTERS)
ref = defined[ref_mask]
if ref.empty:
    print("[WARNING] No core reference clusters found; skipping tumor vs reference Fisher tests.")
    tumor_vs_ref = pd.DataFrame()
else:
    ref_a = int((ref["copykat_pred_clean_for_stats"] == "aneuploid").sum())
    ref_d = int((ref["copykat_pred_clean_for_stats"] == "diploid").sum())
    records = []
    target_clusters = sorted(MALIGNANT_CLUSTERS | CYCLING_CANDIDATE_CLUSTERS, key=lambda x: int(x) if x.isdigit() else 999)
    for cl in target_clusters:
        sub = defined[defined["cluster_str"] == cl]
        if sub.empty:
            continue
        a = int((sub["copykat_pred_clean_for_stats"] == "aneuploid").sum())
        d = int((sub["copykat_pred_clean_for_stats"] == "diploid").sum())
        odds, p = fisher_exact([[a, d], [ref_a, ref_d]], alternative="two-sided")
        records.append({
            "target_cluster": cl,
            "target_role": "cycling_candidate" if cl in CYCLING_CANDIDATE_CLUSTERS else "malignant_like",
            "target_n_defined": a+d,
            "target_n_aneuploid": a,
            "target_n_diploid": d,
            "target_aneuploid_fraction_defined": a/(a+d) if (a+d)>0 else np.nan,
            "reference_clusters": ",".join(sorted(CORE_REFERENCE_CLUSTERS)),
            "reference_n_defined": ref_a+ref_d,
            "reference_n_aneuploid": ref_a,
            "reference_n_diploid": ref_d,
            "reference_aneuploid_fraction_defined": ref_a/(ref_a+ref_d) if (ref_a+ref_d)>0 else np.nan,
            "odds_ratio_target_vs_reference": safe_odds_ratio(a, d, ref_a, ref_d),
            "fisher_p_value": p,
        })
    tumor_vs_ref = pd.DataFrame(records)
    tumor_vs_ref["fisher_q_value_bh"] = bh_fdr(tumor_vs_ref["fisher_p_value"])

write_tsv(tumor_vs_ref, METADATA_DIR / f"final_stat_tumor_vs_reference_copykat_fisher_{RUN_TAG}.tsv")
display(tumor_vs_ref)

if tumor_vs_ref.shape != (5, 14):
    raise RuntimeError(
        "Historical tumor-vs-reference table checkpoint failed: "
        f"{tumor_vs_ref.shape} != (5, 14)"
    )

expected_targets = {
    "0",
    "2",
    "4",
    "7",
    "11",
}

observed_targets = set(
    tumor_vs_ref[
        "target_cluster"
    ]
    .astype(str)
)

if observed_targets != expected_targets:
    raise RuntimeError(
        "Historical tumor-vs-reference target-cluster checkpoint failed. "
        f"Observed: {sorted(observed_targets)}"
    )

print(
    "PASS: historical tumor-vs-reference checkpoint matches exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_stat_tumor_vs_reference_copykat_fisher_HVG8000_r0_6.tsv shape=(5, 14)


,target_cluster,target_role,target_n_defined,target_n_aneuploid,target_n_diploid,target_aneuploid_fraction_defined,reference_clusters,reference_n_defined,reference_n_aneuploid,reference_n_diploid,reference_aneuploid_fraction_defined,odds_ratio_target_vs_reference,fisher_p_value,fisher_q_value_bh
0,0,malignant_like,54857,45910,8947,0.836903,"14,8",16270,3231,13039,0.198586,20.707954,0.0,0.0
1,2,cycling_candidate,35009,29096,5913,0.831101,"14,8",16270,3231,13039,0.198586,19.857873,0.0,0.0
2,4,malignant_like,29316,24754,4562,0.844385,"14,8",16270,3231,13039,0.198586,21.897646,0.0,0.0
3,7,malignant_like,19432,14292,5140,0.735488,"14,8",16270,3231,13039,0.198586,11.221146,0.0,0.0
4,11,malignant_like,5030,3646,1384,0.724851,"14,8",16270,3231,13039,0.198586,10.631337,0.0,0.0


PASS: historical tumor-vs-reference checkpoint matches exactly.


## 6. Sample-level summaries and condition-level tests

This section avoids using only per-cell p-values by aggregating cell states at sample level first.

In [8]:
# Prepare sample-level data.
if sample_col is None:
    obs["sample_for_stats"] = "all_cells_single_sample_missing_sample_id"
    sample_col_use = "sample_for_stats"
else:
    sample_col_use = sample_col

condition_col_use = condition_col if condition_col is not None else None
dataset_col_use = dataset_col if dataset_col is not None else None
patient_col_use = patient_col if patient_col is not None else None

groupby_cols = [sample_col_use]
for c in [patient_col_use, dataset_col_use, condition_col_use]:
    if c is not None and c not in groupby_cols:
        groupby_cols.append(c)

sample_counts = (
    obs.groupby(groupby_cols + ["final_group_for_stats"], observed=True)
       .size().reset_index(name="n_cells")
)
sample_totals = obs.groupby(groupby_cols, observed=True).size().reset_index(name="sample_n_cells")
sample_props = sample_counts.merge(sample_totals, on=groupby_cols, how="left")
sample_props["fraction_of_sample"] = sample_props["n_cells"] / sample_props["sample_n_cells"]
write_tsv(sample_props, METADATA_DIR / f"final_CNV_QC_sample_group_proportions_{RUN_TAG}.tsv")
display(sample_props.head(20))

# Condition-level tests on sample-level proportions.
condition_tests = []
if condition_col_use is not None:
    for gr, sub in sample_props.groupby("final_group_for_stats", observed=True):
        cond_groups = []
        cond_names = []
        for cond, ss in sub.groupby(condition_col_use, observed=True):
            vals = ss["fraction_of_sample"].dropna().astype(float).values
            if len(vals) > 0:
                cond_groups.append(vals)
                cond_names.append(str(cond))
        if len(cond_groups) >= 2:
            # Overall Kruskal-Wallis if at least two condition groups have data.
            try:
                stat, p = kruskal(*cond_groups)
            except Exception:
                stat, p = np.nan, np.nan
            condition_tests.append({
                "test_type": "kruskal_overall_by_condition",
                "final_group": gr,
                "condition_a": "ALL",
                "condition_b": "ALL",
                "n_condition_groups": len(cond_groups),
                "n_samples_total": int(sum(len(x) for x in cond_groups)),
                "statistic": stat,
                "p_value": p,
            })
            # Pairwise Mann-Whitney.
            for i in range(len(cond_groups)):
                for j in range(i+1, len(cond_groups)):
                    va, vb = cond_groups[i], cond_groups[j]
                    if len(va) >= 2 and len(vb) >= 2:
                        try:
                            stat, p = mannwhitneyu(va, vb, alternative="two-sided")
                        except Exception:
                            stat, p = np.nan, np.nan
                        condition_tests.append({
                            "test_type": "mannwhitney_pairwise_by_condition",
                            "final_group": gr,
                            "condition_a": cond_names[i],
                            "condition_b": cond_names[j],
                            "n_condition_groups": 2,
                            "n_samples_total": int(len(va) + len(vb)),
                            "statistic": stat,
                            "p_value": p,
                        })
else:
    print("[SKIP] No condition column found; sample-level condition tests skipped.")

condition_tests = pd.DataFrame(condition_tests)
if not condition_tests.empty:
    condition_tests["q_value_bh"] = bh_fdr(condition_tests["p_value"])
write_tsv(condition_tests, METADATA_DIR / f"final_stat_condition_sample_level_tests_{RUN_TAG}.tsv")
display(condition_tests.head(50))

EXPECTED_SAMPLE_TABLE_SHAPES = {
    "sample_props": (319, 8),
    "condition_tests": (28, 9),
}

observed_sample_table_shapes = {
    "sample_props": sample_props.shape,
    "condition_tests": condition_tests.shape,
}

if observed_sample_table_shapes != EXPECTED_SAMPLE_TABLE_SHAPES:
    raise RuntimeError(
        "Historical Notebook-16 sample-level table shape checkpoint failed. "
        f"Observed: {observed_sample_table_shapes}"
    )

print(
    "PASS: historical sample-level table shapes match exactly."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_CNV_QC_sample_group_proportions_HVG8000_r0_6.tsv shape=(319, 8)


,sample_id,patient_id,core_dataset,condition,final_group_for_stats,n_cells,sample_n_cells,fraction_of_sample
0,GSM5518596_rGBM-01-A,01,DS3_GSE182109,rGBM,cycling_malignant_like_candidate,46,4324,0.010638
1,GSM5518596_rGBM-01-A,01,DS3_GSE182109,rGBM,malignant_like,123,4324,0.028446
2,GSM5518596_rGBM-01-A,01,DS3_GSE182109,rGBM,non_malignant_or_diploid_leaning,46,4324,0.010638
3,GSM5518596_rGBM-01-A,01,DS3_GSE182109,rGBM,review,4109,4324,0.950278
4,GSM5518597_rGBM-01-B,01,DS3_GSE182109,rGBM,cycling_malignant_like_candidate,144,10960,0.013139
5,GSM5518597_rGBM-01-B,01,DS3_GSE182109,rGBM,malignant_like,316,10960,0.028832
6,GSM5518597_rGBM-01-B,01,DS3_GSE182109,rGBM,non_malignant_or_diploid_leaning,211,10960,0.019252
7,GSM5518597_rGBM-01-B,01,DS3_GSE182109,rGBM,review,10289,10960,0.938777
8,GSM5518598_rGBM-01-C,01,DS3_GSE182109,rGBM,cycling_malignant_like_candidate,12,5653,0.002123
9,GSM5518598_rGBM-01-C,01,DS3_GSE182109,rGBM,malignant_like,29,5653,0.005130


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_stat_condition_sample_level_tests_HVG8000_r0_6.tsv shape=(28, 9)


,test_type,final_group,condition_a,condition_b,n_condition_groups,n_samples_total,statistic,p_value,q_value_bh
0,kruskal_overall_by_condition,cycling_malignant_like_candidate,ALL,ALL,4,75,4.252584,0.235450,0.595228
1,mannwhitney_pairwise_by_condition,cycling_malignant_like_candidate,GBM,LGG,2,43,127.000000,0.039332,0.137663
2,mannwhitney_pairwise_by_condition,cycling_malignant_like_candidate,GBM,ndGBM,2,55,329.000000,0.759783,0.868329
3,mannwhitney_pairwise_by_condition,cycling_malignant_like_candidate,GBM,rGBM,2,55,335.000000,0.676713,0.861271
4,mannwhitney_pairwise_by_condition,cycling_malignant_like_candidate,LGG,ndGBM,2,20,10.000000,0.038803,0.137663
5,mannwhitney_pairwise_by_condition,cycling_malignant_like_candidate,LGG,rGBM,2,20,18.000000,0.211352,0.591785
6,mannwhitney_pairwise_by_condition,cycling_malignant_like_candidate,ndGBM,rGBM,2,32,130.000000,0.954917,0.954917
7,kruskal_overall_by_condition,malignant_like,ALL,ALL,4,80,12.591980,0.005607,0.031402
8,mannwhitney_pairwise_by_condition,malignant_like,GBM,LGG,2,43,87.000000,0.731805,0.868329
9,mannwhitney_pairwise_by_condition,malignant_like,GBM,ndGBM,2,59,555.000000,0.008436,0.039367


PASS: historical sample-level table shapes match exactly.


## 7. Output manifest

In [9]:
output_paths = [
    METADATA_DIR / f"final_CNV_QC_readiness_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_CNV_QC_obs_column_report_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_CNV_QC_group_counts_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_CNV_QC_cluster_group_copykat_summary_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_CNV_QC_sample_group_proportions_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_stat_overall_association_tests_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_stat_cluster_copykat_enrichment_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_stat_group_copykat_enrichment_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_stat_tumor_vs_reference_copykat_fisher_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_stat_condition_sample_level_tests_{RUN_TAG}.tsv",
    METADATA_DIR / f"final_CNV_QC_figure_manifest_{RUN_TAG}.tsv",
]
manifest = pd.DataFrame([
    {"output_label": p.stem, "path": str(p), "exists": p.exists(), "size_mb": p.stat().st_size/1024/1024 if p.exists() else np.nan}
    for p in output_paths
])
write_tsv(manifest, METADATA_DIR / f"final_CNV_QC_statistical_output_manifest_{RUN_TAG}.tsv")
display(manifest)
print("DONE")

if manifest.shape != (11, 4):
    raise RuntimeError(
        "Historical Notebook-16 statistical-output manifest checkpoint failed: "
        f"{manifest.shape} != (11, 4)"
    )

if not manifest[
    "exists"
].all():
    raise RuntimeError(
        "One or more expected Notebook-16 outputs are missing."
    )

print(
    "\nFINAL CHECKPOINTS"
)
print(
    "Canonical final H5AD:",
    FINAL_H5AD,
)
print(
    "Source shape:",
    source_shape,
)
print(
    "Canonical CopyKAT counts:",
    EXPECTED_COPYKAT_COUNTS,
)
print(
    "Final malignancy-group counts:",
    EXPECTED_FINAL_GROUP_COUNTS,
)
print(
    "Group counts:",
    group_counts.shape,
)
print(
    "Cluster/group summary:",
    cluster_group_summary.shape,
)
print(
    "Figure manifest:",
    figure_manifest.shape,
)
print(
    "Defined CopyKAT cells:",
    len(defined),
)
print(
    "Overall association tests:",
    overall_tests.shape,
)
print(
    "Cluster CopyKAT enrichment:",
    cluster_enrichment.shape,
)
print(
    "Group CopyKAT enrichment:",
    group_enrichment.shape,
)
print(
    "Tumor vs reference Fisher:",
    tumor_vs_ref.shape,
)
print(
    "Sample/group proportions:",
    sample_props.shape,
)
print(
    "Condition sample-level tests:",
    condition_tests.shape,
)
print(
    "Statistical output manifest:",
    manifest.shape,
)
print(
    "\n[DONE] Notebook 16 canonical CNV validation completed."
)


[SAVED] G:\Repository\glioma-cnv-single-cell-atlas\metadata\final_CNV_QC_statistical_output_manifest_HVG8000_r0_6.tsv shape=(11, 4)


,output_label,path,exists,size_mb
0,final_CNV_QC_readiness_HVG8000_r0_6,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000314
1,final_CNV_QC_obs_column_report_HVG8000_r0_6,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000258
2,final_CNV_QC_group_counts_HVG8000_r0_6,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000836
3,final_CNV_QC_cluster_group_copykat_summary_HVG...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001607
4,final_CNV_QC_sample_group_proportions_HVG8000_...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.029026
5,final_stat_overall_association_tests_HVG8000_r0_6,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000220
6,final_stat_cluster_copykat_enrichment_HVG8000_...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.001782
7,final_stat_group_copykat_enrichment_HVG8000_r0_6,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000456
8,final_stat_tumor_vs_reference_copykat_fisher_H...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.000871
9,final_stat_condition_sample_level_tests_HVG800...,G:\Repository\glioma-cnv-single-cell-atlas\met...,True,0.003230


DONE

FINAL CHECKPOINTS
Canonical final H5AD: G:\Repository\glioma-cnv-single-cell-atlas\data\processed\integrated\GBM_core_4datasets_fullgene_marker_based_annotation_bias_status_copykat_consensus.h5ad
Source shape: (351427, 15176)
Canonical CopyKAT counts: {'aneuploid': 202895, 'diploid': 111493, 'not_defined': 37037, 'missing': 2}
Final malignancy-group counts: {'review': 167178, 'malignant_like': 123250, 'cycling_malignant_like_candidate': 38634, 'non_malignant_or_diploid_leaning': 22365}
Group counts: (14, 4)
Cluster/group summary: (18, 10)
Figure manifest: (7, 4)
Defined CopyKAT cells: 314388
Overall association tests: (2, 8)
Cluster CopyKAT enrichment: (18, 10)
Group CopyKAT enrichment: (4, 8)
Tumor vs reference Fisher: (5, 14)
Sample/group proportions: (319, 8)
Condition sample-level tests: (28, 9)
Statistical output manifest: (11, 4)

[DONE] Notebook 16 canonical CNV validation completed.
